In [9]:
models = [
        "ARIMA",
        "ETS",
        "THETA",
        "svr",
        "rf",
        "catboost",
        "CWT_svr",
        "DWT_svr",
        "FT_svr",
        "CWT_rf",
        "DWT_rf",
        "FT_rf",
        "CWT_catboost",
        "DWT_catboost",
        "FT_catboost",
        "ONLY_CWT_catboost",
        "ONLY_CWT_rf",
        "ONLY_CWT_svr",
        "ONLY_DWT_catboost",
        "ONLY_DWT_rf",
        "ONLY_DWT_svr",
        "ONLY_FT_catboost",
        "ONLY_FT_rf",
        "ONLY_FT_svr",
        "NaiveSeasonal",
        "NaiveMovingAverage",
    ]

In [10]:
import re
def read_model_preds(model_name, dataset_index):
    try:
        import pandas as pd
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "pandas is required to read model prediction CSVs. Install it in your environment (e.g., `pip install pandas`)."
        ) from e

    df = pd.read_csv(
        f"../timeseries/mestrado/resultados/{model_name}/normal/ANP_MONTHLY.csv",
        sep=";",
    )
    df = df[df["dataset_index"] == dataset_index]

    df["start_test"] = pd.to_datetime(df["start_test"], format="%Y-%m-%d")
    df["final_test"] = pd.to_datetime(df["final_test"], format="%Y-%m-%d")
    df = df.sort_values(by="start_test")

    return df

def extract_values(list_str):
    if isinstance(list_str, str):
        numbers = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", list_str)
        return [float(num) for num in numbers]
    return []

def generate_validations(models, dataset_index):
    sample_model = models[0]
    df_sample = read_model_preds(sample_model, dataset_index)
    df_filtred_sample = df_sample.iloc[-4:-1]
    n_windows = len(df_filtred_sample)

    all_validations = {"test": [], "predictions": []}
    for _ in range(n_windows):
        all_validations["predictions"].append({})
        
    test_extracted = False  
    final_test_predictions = {}
    for model in models:  
        df_model = read_model_preds(model, dataset_index)
        df_filtred = df_model.iloc[-4:-1]
        df_final_test = df_model.iloc[-1]
        predictions_final = extract_values(df_final_test["predictions"])
        final_test_predictions[model] = predictions_final
        for window_idx, (_, row) in enumerate(df_filtred.iterrows()):
            preds = extract_values(row["predictions"])
            all_validations["predictions"][window_idx][model] = preds
            
            if not test_extracted: 
                test = extract_values(row["test"])
                all_validations["test"].append(test)
        
        if not test_extracted: 
            test_extracted = True
            
    return all_validations, final_test_predictions

In [11]:
all_validations, final_test_predictions = generate_validations(models, dataset_index=0)

In [14]:
import os
import numpy as np
import matplotlib.pyplot as plt


def _as_float_array(x):
    return np.asarray(x, dtype=float)


def _extract_models(pred_dict):
    # pred_dict: dict {model_name: [..]}
    return list(pred_dict.keys())


def _sliding_window_errors(y_true, y_pred, H=3):
    """
    Retorna erros por janela e horizonte.
    y_true: (T,)
    y_pred: (T,)
    H: int (ex.: 3)
    returns:
      err_wh: shape (n_windows, H) com erro assinado (yhat - y)
      abs_err_wh: shape (n_windows, H)
    """
    y_true = _as_float_array(y_true)
    y_pred = _as_float_array(y_pred)
    T = len(y_true)
    if len(y_pred) != T:
        raise ValueError(f"y_pred len {len(y_pred)} != y_true len {T}")

    n_windows = T - H + 1
    if n_windows <= 0:
        raise ValueError(f"T={T} é pequeno demais para H={H}")

    err_wh = np.zeros((n_windows, H), dtype=float)
    for w in range(n_windows):
        yt = y_true[w : w + H]
        yp = y_pred[w : w + H]
        err_wh[w, :] = (yp - yt)
    abs_err_wh = np.abs(err_wh)
    return err_wh, abs_err_wh


def _robust_denom(y_true):
    y_true = _as_float_array(y_true)
    eps = 1e-8
    return np.mean(np.abs(y_true)) + eps


def _combine_fold_images_matplotlib(image_paths, out_path, direction="vertical", padding_px=30):
    """
    Combina PNGs já gerados em uma imagem única (mosaico) usando matplotlib (sem Pillow).
    direction: "vertical" ou "horizontal"
    """
    imgs = [plt.imread(p) for p in image_paths]

    heights = [im.shape[0] for im in imgs]
    widths = [im.shape[1] for im in imgs]

    if direction == "vertical":
        out_h = sum(heights) + padding_px * (len(imgs) - 1)
        out_w = max(widths)
        canvas = np.ones((out_h, out_w, 3), dtype=float)  # fundo branco

        y = 0
        for im in imgs:
            h, w = im.shape[0], im.shape[1]
            x = (out_w - w) // 2
            rgb = im[..., :3]  # descarta alpha se existir
            canvas[y : y + h, x : x + w, :] = rgb
            y += h + padding_px

    elif direction == "horizontal":
        out_w = sum(widths) + padding_px * (len(imgs) - 1)
        out_h = max(heights)
        canvas = np.ones((out_h, out_w, 3), dtype=float)

        x = 0
        for im in imgs:
            h, w = im.shape[0], im.shape[1]
            y = (out_h - h) // 2
            rgb = im[..., :3]
            canvas[y : y + h, x : x + w, :] = rgb
            x += w + padding_px
    else:
        raise ValueError("direction deve ser 'vertical' ou 'horizontal'")

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    plt.imsave(out_path, canvas)
    return out_path


def plot_fold_figure(
    fold_idx,
    y_true,
    preds_by_model,
    out_path,
    H=3,
    vlim=2.0,
):
    """
    Gera figura por fold otimizada para LLM Multimodal.

    Painéis gerados:
      A: Gráfico de Barras Horizontal (Ranking de MAE)
      B: Scatter Plot (Viés vs MAE) com nomes dos modelos
      C: Heatmap de MAE por horizonte com valores anotados
      D: y_true vs TODOS os modelos
    """
    y_true = _as_float_array(y_true)
    models = sorted(preds_by_model.keys())

    denom = _robust_denom(y_true)

    mean_signed = []
    mae_norm = []
    overall_mae = []
    overall_bias = []

    for m in models:
        err_wh, abs_err_wh = _sliding_window_errors(y_true, preds_by_model[m], H=H)

        mean_signed_err_h = np.mean(err_wh, axis=0) / denom
        mae_h = np.mean(abs_err_wh, axis=0) / denom

        mean_signed.append(mean_signed_err_h)
        mae_norm.append(mae_h)
        overall_mae.append(np.mean(mae_h))
        overall_bias.append(np.mean(mean_signed_err_h))

    mean_signed = np.vstack(mean_signed)  # (M,H)
    mae_norm = np.vstack(mae_norm)        # (M,H)
    overall_mae = np.asarray(overall_mae)
    overall_bias = np.asarray(overall_bias)

    order = np.argsort(overall_mae)
    models_ord = [models[i] for i in order]
    mean_signed_ord = mean_signed[order]
    mae_norm_ord = mae_norm[order]
    overall_mae_ord = overall_mae[order]
    overall_bias_ord = overall_bias[order]

    fig = plt.figure(figsize=(22, 16))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1])

    # Painel A: Gráfico de Barras Horizontal (Ranking de MAE)
    axA = fig.add_subplot(gs[0, 0])
    y_pos = np.arange(len(models_ord))
    # Inverter para o melhor (menor MAE) ficar no topo
    axA.barh(y_pos[::-1], overall_mae_ord, color='skyblue', edgecolor='black')
    axA.set_yticks(y_pos[::-1])
    axA.set_yticklabels(models_ord, fontsize=9)
    axA.set_xlabel("MAE Médio Normalizado")
    axA.set_title(f"Fold {fold_idx} — Painel A: Ranking de Modelos (Menor MAE é melhor)")
    for i, v in enumerate(overall_mae_ord):
        axA.text(v + 0.01, y_pos[::-1][i], f"{v:.3f}", va='center', fontsize=8)

    # Painel B: Scatter Plot (Viés vs MAE)
    axB = fig.add_subplot(gs[0, 1])
    axB.scatter(overall_mae_ord, overall_bias_ord, color='coral', s=50, edgecolor='black')
    axB.axhline(0, color='gray', linestyle='--', linewidth=1)
    axB.set_xlabel("MAE Médio Normalizado (Erro Absoluto)")
    axB.set_ylabel("Viés Médio Normalizado (Erro com Sinal)")
    axB.set_title(f"Fold {fold_idx} — Painel B: Diversidade (Viés vs MAE)")
    
    # Anotar os nomes dos modelos nos pontos
    for i, txt in enumerate(models_ord):
        axB.annotate(txt, (overall_mae_ord[i], overall_bias_ord[i]), 
                     xytext=(5, 0), textcoords='offset points', fontsize=8, alpha=0.8)

    # Painel C: Heatmap de MAE por horizonte com valores anotados
    axC = fig.add_subplot(gs[1, 0])
    imC = axC.imshow(mae_norm_ord, aspect="auto", cmap="viridis", interpolation="nearest")
    axC.set_title(f"Fold {fold_idx} — Painel C: MAE por Horizonte (Valores Anotados)")
    axC.set_yticks(np.arange(len(models_ord)))
    axC.set_yticklabels(models_ord, fontsize=9)
    axC.set_xticks(np.arange(H))
    axC.set_xticklabels([f"H{i+1}" for i in range(H)])
    cbarC = fig.colorbar(imC, ax=axC, fraction=0.046, pad=0.04)
    cbarC.set_label("MAE normalizado")
    
    # Anotar os valores dentro do heatmap
    for i in range(len(models_ord)):
        for j in range(H):
            val = mae_norm_ord[i, j]
            color = "white" if val < (np.max(mae_norm_ord) / 2) else "black"
            axC.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=8)

    # Painel D: y_true vs TODOS os modelos
    axD = fig.add_subplot(gs[1, 1])
    t = np.arange(len(y_true))
    
    axD.plot(t, y_true, color="black", linewidth=3.0, label="y_true", zorder=10)

    # Usar um colormap para diferenciar as linhas
    cmap = plt.get_cmap("tab20")
    colors = [cmap(i % 20) for i in range(len(models_ord))]
    
    for i, m in enumerate(models_ord):
        axD.plot(t, _as_float_array(preds_by_model[m]), linewidth=1.5, alpha=0.7, label=m, color=colors[i])

    axD.set_title(f"Fold {fold_idx} — Painel D: y_true vs TODOS os modelos (T={len(y_true)})")
    axD.set_xlabel("t (índice no período de validação)")
    
    # Colocar a legenda fora do gráfico para não sobrepor as linhas
    axD.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=8, ncol=2)

    fig.tight_layout()
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.close(fig)

    return {
        "models_ordered": models_ord,
        "overall_mae_norm_ordered": overall_mae_ord.tolist(),
        "mean_signed_err_norm_ordered": mean_signed_ord.tolist(),
        "mae_norm_ordered": mae_norm_ord.tolist(),
    }


def plot_from_payload(
    payload,
    out_dir="figs_out",
    H=3,
    combine_folds=True,
    combined_filename=None,
    combine_direction="vertical",
):
    """
    payload esperado:
    {
      "test": [list_of_y_true_fold0, list_of_y_true_fold1, ...],
      "predictions": [ {model: y_pred_list, ...}, {..}, ...]
    }

    Além de gerar fold_{k}_H{H}.png, opcionalmente gera um mosaico único
    combinando todos folds (ex.: all_folds_H3.png).
    """
    if "test" not in payload or "predictions" not in payload:
        raise ValueError("payload precisa ter as chaves: 'test' e 'predictions'")

    tests = payload["test"]
    preds = payload["predictions"]

    if not isinstance(tests, list) or not isinstance(preds, list):
        raise ValueError("'test' e 'predictions' precisam ser listas (um item por fold/janela).")

    if len(tests) != len(preds):
        raise ValueError(f"len(test)={len(tests)} != len(predictions)={len(preds)}")

    model_sets = [set(_extract_models(p)) for p in preds]
    common_models = set.intersection(*model_sets)
    if not common_models:
        raise ValueError("Não há modelos em comum entre os folds em payload['predictions'].")
    common_models = sorted(common_models)

    os.makedirs(out_dir, exist_ok=True)

    results = []
    fold_paths = []

    for k, (y_true, pred_dict) in enumerate(zip(tests, preds)):
        preds_by_model = {m: pred_dict[m] for m in common_models}

        out_path = os.path.join(out_dir, f"fold_{k}_H{H}.png")
        res = plot_fold_figure(
            fold_idx=k,
            y_true=y_true,
            preds_by_model=preds_by_model,
            out_path=out_path,
            H=H,
            vlim=2.0,
        )
        results.append({"fold": k, "out_path": out_path, **res})
        fold_paths.append(out_path)

    combined_path = None
    if combine_folds and len(fold_paths) >= 2:
        if combined_filename is None:
            combined_filename = f"all_folds_H{H}.png"
        combined_path = os.path.join(out_dir, combined_filename)
        _combine_fold_images_matplotlib(
            image_paths=fold_paths,
            out_path=combined_path,
            direction=combine_direction,
            padding_px=30,
        )

    return {
        "folds": results,
        "combined_path": combined_path,
        "fold_image_paths": fold_paths,
        "common_models": common_models,
    }


# Exemplo de uso:
# out = plot_from_payload(payload, out_dir="figs_out", H=3, combine_folds=True)
# print(out["combined_path"])

In [15]:
plot_from_payload(all_validations, out_dir="fis_2", H=3)

{'folds': [{'fold': 0,
   'out_path': 'fis_2/fold_0_H3.png',
   'models_ordered': ['ARIMA',
    'FT_svr',
    'ETS',
    'ONLY_CWT_rf',
    'THETA',
    'ONLY_FT_rf',
    'NaiveMovingAverage',
    'ONLY_FT_catboost',
    'svr',
    'ONLY_DWT_svr',
    'ONLY_FT_svr',
    'CWT_svr',
    'catboost',
    'ONLY_DWT_rf',
    'NaiveSeasonal',
    'FT_rf',
    'rf',
    'ONLY_DWT_catboost',
    'DWT_rf',
    'CWT_rf',
    'DWT_svr',
    'DWT_catboost',
    'FT_catboost',
    'ONLY_CWT_catboost',
    'CWT_catboost',
    'ONLY_CWT_svr'],
   'overall_mae_norm_ordered': [0.23589121417823686,
    0.2444713716968542,
    0.25268317124646716,
    0.28225135602441864,
    0.28437718850102495,
    0.28461120376296595,
    0.2908636724321873,
    0.29820644281602326,
    0.3115117612671063,
    0.312887915506064,
    0.31736819996876847,
    0.321200443998223,
    0.32187274150458317,
    0.33407092330496013,
    0.337612286095656,
    0.3392776676126284,
    0.3415738235364625,
    0.34207636255075036,